In [1]:
import pandas as pd
import numpy as np


import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score

sns.set(style='white', context='notebook')


In [ ]:
df_final = pd.read_csv('adult.csv')
df_final.head()

In [ ]:
df_final = df_final.reset_index(drop=True)
df_final.head()

In [ ]:
df_final= df_final[df_final.occupation != '?']
print (df_final['occupation'].value_counts())

In [ ]:
def pick_col(df_final, candidates):
    for c in candidates:
        if c in df_final.columns:
            return c
    raise KeyError(f"None of {candidates} found in DataFrame columns.")

race_col  = pick_col(df_final, ["race"])
sex_col   = pick_col(df_final, ["sex"])
work_col  = pick_col(df_final, ["workclass", "work-class"])
occ_col   = pick_col(df_final, ["occupation"])
educ_col  = pick_col(df_final, ["education"])
educn_col = pick_col(df_final, ["education.num", "education_num"])
ms_col    = pick_col(df_final, ["marital-status", "marital.status"])
hpw_col   = pick_col(df_final, ["hours.per.week", "hours_per_week"])
age_col   = pick_col(df_final, ["age"])
cg_col    = pick_col(df_final, ["capital.gain", "capital_gain"])
cl_col    = pick_col(df_final, ["capital.loss", "capital_loss"])
inc_col   = pick_col(df_final, ["income"])


for c in [race_col, sex_col, work_col, occ_col, educ_col, ms_col, inc_col]:
    if c in df_final.columns and df_final[c].dtype == object:
        df_final[c] = df_final[c].astype(str).str.strip()
df_final[inc_col] = df_final[inc_col].str.replace('.', '', regex=False)


y_num = (df_final[inc_col] == ">50K").astype(int)

def plot_rate_bar(df_final, cat_col, target_col=inc_col, top_n=None, title=None, rotate=0):
    tmp = (df_final.dropna(subset=[cat_col, target_col])
             .groupby(cat_col)[target_col]
             .agg(rate=lambda s: (s == ">50K").mean(), count="size")
             .sort_values("rate", ascending=False))
    if top_n:
        tmp = tmp.head(top_n)
    ax = tmp["rate"].plot(kind="barh", figsize=(10,6))
    ax.invert_yaxis()
    ax.set_xlabel("P(income > 50K)")
    ax.set_ylabel(cat_col)
    ax.set_title(title or f"High-income rate by {cat_col}")
    
    for i,(rate,row) in enumerate(zip(tmp["rate"], tmp.itertuples())):
        ax.text(rate+0.005, i, f" n={row.count}", va="center")
    plt.tight_layout()
    return ax

def target_rate_heatmap(df_final, row, col, target=inc_col, min_count=50, title=None, cmap="Greens"):
    g = (df_final.dropna(subset=[row,col,target])
           .groupby([row,col])[target]
           .agg(count="size", rate=lambda s: (s==">50K").mean())
           .reset_index())
    
    g = g[g["count"] >= min_count]
    pt = g.pivot(index=row, columns=col, values="rate")
    plt.figure(figsize=(10,6))
    sns.heatmap(pt, cmap=cmap, annot=True, fmt=".2f", cbar_kws={"shrink": .8})
    plt.title(title or f"High-income rate by {row} × {col}", fontsize=16, fontweight="bold")
    plt.xlabel(col); plt.ylabel(row)
    plt.tight_layout()
    return pt

In [ ]:
#High income rate by Education

edu_order = ['Preschool','1st-4th','5th-6th','7th-8th','9th','10th','11th','12th',
             'HS-grad','Some-college','Assoc-voc','Assoc-acdm','Bachelors','Masters','Prof-school','Doctorate']
df_final[educ_col] = pd.Categorical(df_final[educ_col], categories=[e for e in edu_order if e in df_final[educ_col].unique()], ordered=True)
plot_rate_bar(df_final, educ_col, title="P(>50K) by Education (sorted)")
plt.show()

In [ ]:
#Income rate by Occupation (Top 15)
occ_counts = df_final[occ_col].value_counts()
top_occ = occ_counts.index[:15]
plot_rate_bar(df_final[df_final[occ_col].isin(top_occ)], occ_col, title="P(>50K) by Occupation (Top 15)")
plt.show()

In [ ]:
#Race x Sex Income rate heatmap
target_rate_heatmap(df_final, row=race_col, col=sex_col, min_count=50,
                    title="High-income Rate by Race × Sex")
plt.show()

In [ ]:
df_final["_age_bin"] = pd.cut(df_final[age_col], bins=[16,25,35,45,55,65,90],
                        labels=["17–25","26–35","36–45","46–55","56–65","66+"])
df_final["_hrs_bin"] = pd.cut(df_final[hpw_col], bins=[0,20,30,40,50,60,100],
                        labels=["≤20","21–30","31–40","41–50","51–60","61+"])
target_rate_heatmap(df_final, row="_age_bin", col="_hrs_bin", min_count=80,
                    title="P(>50K) by Age Bin × Hours/Week")
plt.show()

In [ ]:


inc_col, race_col, sex_col = "income", "race", "sex"
df_final[inc_col] = df_final[inc_col].astype(str).str.strip().str.replace(".", "", regex=False)
df_final[race_col] = df_final[race_col].astype(str).str.strip()
df_final[sex_col] = df_final[sex_col].astype(str).str.strip()

min_count = 50  
g = (
    df_final.dropna(subset=[race_col, sex_col, inc_col])
      .assign(target=(df_final[inc_col] == ">50K").astype(int))
      .groupby([race_col, sex_col])["target"]
      .agg(rate="mean", count="size")
      .reset_index()
)
g = g[g["count"] >= min_count]

pt_rate = g.pivot(index=race_col, columns=sex_col, values="rate")
pt_cnt  = g.pivot(index=race_col, columns=sex_col, values="count")

race_order = [r for r in ["White", "Black", "Asian-Pac-Islander", "Amer-Indian-Eskimo", "Other"] if r in pt_rate.index]
sex_order  = [s for s in ["Female", "Male"] if s in pt_rate.columns]

pt_rate = pt_rate.reindex(index=race_order or pt_rate.index, columns=sex_order or pt_rate.columns)
pt_cnt  = pt_cnt.reindex(index=pt_rate.index, columns=pt_rate.columns)

pt_pct = (pt_rate * 100).round(0)
labels = pt_pct.astype("Int64").astype(str) + "%\n(n=" + pt_cnt.astype("Int64").astype(str) + ")"

plt.figure(figsize=(8.5, 5.5))
ax = sns.heatmap(
    pt_pct,
    cmap="Greens",
    annot=labels,
    fmt="",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"shrink": 0.8}
)
ax.set_title("High-income rate by Race × Sex", fontsize=14, fontweight="bold")
ax.set_xlabel("Sex")
ax.set_ylabel("Race")


cbar = ax.collections[0].colorbar
cbar.set_label("P(>50K) %")

plt.tight_layout()
